# 02 — Player Transfer Metrics Review

**Source:** 358 columns from `male_transfer_model.parquet` + `transfers_same_competitions_data.parquet`  
**Goal:** Understand the 3 metric representations (Qualities, Per-90, Z-scores), their overlap, coverage, and decide which to keep.

---

### What's in this file

| Sub-group | FROM | TO | Total | Description |
|-----------|------|----|-------|-------------|
| Structural | — | — | 16 | IDs, position, season, minutes |
| **Qualities** (composites) | 29 | 29 | 58 | Twelve's aggregated player skill indices |
| **Per-90 / rate stats** | 67 | 67 | 134 | Normalized per-90-min or percentage metrics |
| **Z-scores** | 75 | 75 | 150 | Position × season standardized scores |
| **Total** | | | **358** | |

### Key questions to answer
1. Are the 3 representations measuring the same things? What overlaps, what doesn't?
2. How correlated are they? (Quality X vs Per-90 Y vs Z-score Y)
3. What's the null coverage for each?
4. What are the z-score cross-season issues?
5. **Decision: which representation(s) to keep?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=0.85)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:.3f}'.format)

BASE = Path.home() / 'Documents'
# Handle macOS iCloud path with special apostrophe
import subprocess
def find_file(name):
    r = subprocess.run(['find', str(BASE), '-name', name], capture_output=True, text=True)
    lines = [l for l in r.stdout.strip().split('\n') if l]
    return Path(lines[0]) if lines else None

PARQUET = find_file('transfers_model_2018_2025.parquet')
# If the file was renamed back to v2, try that too
if PARQUET is None:
    PARQUET = find_file('transfers_model_v2_2018_2025.parquet')
print(f'Loading: {PARQUET}')
df = pd.read_parquet(PARQUET)
print(f'Shape: {df.shape[0]:,} × {df.shape[1]}')

---
## 1. The three representations side by side

Let's separate the FROM-side metrics into their 3 groups and see what each one contains.  
(TO-side is a perfect mirror, so we only need to inspect FROM.)

In [ ]:
# Separate the 3 representations (FROM side only — TO is symmetric)
qualities = sorted([c for c in df.columns if c.startswith('from_') 
                    and 'z_score' not in c and 'per 90' not in c and 'team_stats' not in c
                    and 'comp_' not in c
                    and c not in ('from_Minutes','from_competition','from_position','from_season','from_team_id',
                                  'from_matches_pct','from_minutes_pct')
                    and not any(x in c for x in ['per 100','per box','per shot','per possession','per opponent','per low'])
                    and not c.endswith(' %')])

per90 = sorted([c for c in df.columns if c.startswith('from_') 
                and 'z_score' not in c and 'team_stats' not in c and 'comp_' not in c
                and c not in ('from_Minutes','from_competition','from_position','from_season','from_team_id')
                and c not in qualities])

zscores = sorted([c for c in df.columns if c.startswith('from_z_score_')])

# Clean names for comparison
qual_names = sorted([c.replace('from_', '') for c in qualities])
p90_names  = sorted([c.replace('from_', '') for c in per90])
zs_names   = sorted([c.replace('from_z_score_', '') for c in zscores])

print(f'Qualities:  {len(qualities)} metrics')
print(f'Per-90/rate: {len(per90)} metrics')
print(f'Z-scores:   {len(zscores)} metrics')
print(f'\n--- Qualities ({len(qual_names)}) ---')
for n in qual_names: print(f'  {n}')
print(f'\n--- Per-90 / rate ({len(p90_names)}) ---')
for n in p90_names: print(f'  {n}')
print(f'\n--- Z-scores ({len(zs_names)}) ---')
for n in zs_names: print(f'  {n}')

## 2. Metric overlap between the 3 representations

Are they measuring the same underlying things? Let's compare names.

In [ ]:
# Find overlaps
p90_set = set(p90_names)
zs_set = set(zs_names)
qual_set = set(qual_names)

# Z-scores that map to a per-90 metric (same name)
zs_in_p90 = zs_set & p90_set
zs_only = zs_set - p90_set
p90_only = p90_set - zs_set

print(f'=== Z-score vs Per-90 overlap ===')
print(f'Shared metric names: {len(zs_in_p90)}')
print(f'Z-score only (no per-90 equivalent): {len(zs_only)}')
print(f'Per-90 only (no z-score equivalent): {len(p90_only)}')

if zs_only:
    print(f'\nZ-score ONLY metrics (these have z-scores but no raw per-90):')
    for n in sorted(zs_only): print(f'  {n}')

if p90_only:
    print(f'\nPer-90 ONLY metrics (raw per-90 but no z-score):')
    for n in sorted(p90_only): print(f'  {n}')

print(f'\n=== Qualities vs the others ===')
qual_in_zs = qual_set & zs_set
qual_in_p90 = qual_set & p90_set
qual_exclusive = qual_set - zs_set - p90_set
print(f'Quality names also in z-scores: {len(qual_in_zs)}')
print(f'Quality names also in per-90: {len(qual_in_p90)}')
print(f'Quality-exclusive metrics: {len(qual_exclusive)}')
if qual_exclusive:
    print('These are ONLY available as Qualities (composites):')
    for n in sorted(qual_exclusive): print(f'  {n}')
if qual_in_zs:
    print(f'\nQualities that also exist as z-scores:')
    for n in sorted(qual_in_zs): print(f'  {n}')

## 3. Coverage / null analysis by representation

In [ ]:
# Build a coverage comparison table
coverage = []

for name, cols, prefix in [
    ('Qualities FROM', qualities, 'from_'),
    ('Per-90 FROM', per90, 'from_'),
    ('Z-scores FROM', zscores, 'from_z_score_'),
]:
    null_rates = df[cols].isnull().mean() * 100
    coverage.append({
        'Representation': name,
        'N metrics': len(cols),
        'Avg null %': null_rates.mean(),
        'Median null %': null_rates.median(),
        'Max null %': null_rates.max(),
        'Min null %': null_rates.min(),
        'Metrics >10% null': (null_rates > 10).sum(),
        'Metrics >50% null': (null_rates > 50).sum(),
    })

cov_df = pd.DataFrame(coverage)
cov_df.style.format({
    'Avg null %': '{:.1f}', 'Median null %': '{:.1f}', 
    'Max null %': '{:.1f}', 'Min null %': '{:.1f}'
}).background_gradient(cmap='YlOrRd', subset=['Avg null %', 'Max null %'])

In [ ]:
# Detailed: which specific metrics have the worst coverage?
all_from_metrics = qualities + per90 + zscores
null_detail = pd.DataFrame({
    'column': all_from_metrics,
    'null_pct': [df[c].isnull().mean() * 100 for c in all_from_metrics],
    'type': (['quality'] * len(qualities) + 
             ['per90'] * len(per90) + 
             ['z_score'] * len(zscores))
})

problematic = null_detail[null_detail['null_pct'] > 5].sort_values('null_pct', ascending=False)
print(f'Metrics with >5% null: {len(problematic)}\n')
problematic

## 4. What are the Qualities, exactly?

Qualities are Twelve's **composite indices** — each one aggregates multiple underlying metrics into a single score. Let's see how they look.

In [ ]:
# Distributions of all 29 qualities
fig, axes = plt.subplots(5, 6, figsize=(20, 14))
axes = axes.flatten()

for i, col in enumerate(qualities):
    if i >= 30: break
    ax = axes[i]
    vals = df[col].dropna()
    ax.hist(vals, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    name = col.replace('from_', '')
    ax.set_title(name, fontsize=8)
    ax.tick_params(labelsize=6)

# Hide unused axes
for j in range(len(qualities), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('FROM Qualities — distributions (Twelve composite indices)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Basic stats for qualities
df[qualities].describe().T.rename(columns=lambda c: c).style.format('{:.2f}')

In [ ]:
# Correlation between qualities — how redundant are they?
fig, ax = plt.subplots(figsize=(14, 12))
corr = df[qualities].corr()
# Clean labels
labels = [c.replace('from_', '') for c in qualities]
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, xticklabels=labels, yticklabels=labels,
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=False,
            ax=ax, square=True)
ax.set_title('Inter-quality correlation matrix')
ax.tick_params(labelsize=7)
plt.tight_layout()
plt.show()

# Flag pairs with |corr| > 0.8
high_corr = []
for i in range(len(qualities)):
    for j in range(i+1, len(qualities)):
        r = corr.iloc[i, j]
        if abs(r) > 0.8:
            high_corr.append((labels[i], labels[j], round(r, 3)))

if high_corr:
    print(f'Quality pairs with |corr| > 0.8: {len(high_corr)}')
    for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f'  {a:40s} ↔ {b:40s}  r={r:+.3f}')
else:
    print('No quality pairs with |corr| > 0.8')

## 5. Per-90 metrics — the granular view

These are the concrete, interpretable numbers.

In [ ]:
# Distributions of all per-90 metrics (sample of 30 for readability)
fig, axes = plt.subplots(6, 6, figsize=(20, 18))
axes = axes.flatten()

# Pick a diverse sample
sample_p90 = per90[::2][:36]  # every other one, up to 36

for i, col in enumerate(sample_p90):
    ax = axes[i]
    vals = df[col].dropna()
    vals = vals[np.isfinite(vals)]  # remove inf/-inf
    if len(vals) > 0:
        ax.hist(vals, bins=40, color='darkorange', edgecolor='white', alpha=0.8)
    name = col.replace('from_', '')
    ax.set_title(name, fontsize=7)
    ax.tick_params(labelsize=6)

for j in range(len(sample_p90), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('FROM Per-90 / rate metrics — distributions (sample)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
df[per90].describe().T.style.format('{:.2f}')

## 6. Z-scores — the position-normalized view

Computed by Twelve **per position × season** (global, all leagues).  
A z-score of +1.0 means: *1 std above the mean for that position in that season across all leagues.*

**Problem:** FROM and TO z-scores use different reference populations (different seasons).  
Let's visualize the issue.

In [ ]:
# Are z-scores actually standardized? Check mean/std by position × season
test_metric = 'from_z_score_Goals per 90'
check = df.groupby(['from_position', 'from_season'])[test_metric].agg(['mean', 'std', 'count'])
check = check[check['count'] > 50]  # only groups with enough data
print(f'Z-score "{test_metric}" — mean & std by (position, season):')
print(f'  Mean of means: {check["mean"].mean():.4f} (should be ~0)')
print(f'  Mean of stds:  {check["std"].mean():.4f} (should be ~1)')
print()
check.head(15)

In [ ]:
# THE CROSS-SEASON PROBLEM:
# For the same position, how much does the underlying per-90 mean shift across seasons?
# If it shifts a lot, z-scores from different seasons are NOT comparable.

test_raw = 'from_Goals per 90'
position_to_check = 'Centre Forward'  # or whatever the most common forward position is

# Find actual position names
top_positions = df['from_position'].value_counts().head(5)
print('Top 5 positions:')
print(top_positions)
print()

In [ ]:
# Show how the RAW per-90 distributions shift across seasons for a few key metrics
# This reveals whether z-score baselines are moving

top_pos = df['from_position'].value_counts().index[0]  # most common position
key_metrics = ['from_Goals per 90', 'from_xG per 90', 'from_Defensive actions per 90', 
               'from_Passes (xT) per 90', 'from_Ball recoveries per 90', 'from_Counterpressing per 90']

# Filter to available metrics
key_metrics = [m for m in key_metrics if m in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

pos_mask = df['from_position'] == top_pos
seasons = sorted(df.loc[pos_mask, 'from_season'].dropna().unique())

for i, metric in enumerate(key_metrics[:6]):
    ax = axes[i]
    means, stds = [], []
    for s in seasons:
        mask = pos_mask & (df['from_season'] == s)
        vals = df.loc[mask, metric].dropna()
        means.append(vals.mean())
        stds.append(vals.std())
    
    ax.errorbar(range(len(seasons)), means, yerr=stds, fmt='o-', capsize=3, color='steelblue')
    ax.set_xticks(range(len(seasons)))
    ax.set_xticklabels(seasons, rotation=45, fontsize=7)
    ax.set_title(metric.replace('from_', ''), fontsize=9)
    ax.set_ylabel('mean ± std')

for j in range(len(key_metrics), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'Raw per-90 distribution shift across seasons — position: {top_pos}\n'
             f'(if these shift, z-scores from different seasons have different baselines)', fontsize=11)
plt.tight_layout()
plt.show()

## 7. How much do Qualities add beyond Per-90?

If qualities are just aggregations of per-90 metrics, they might be redundant. Let's check how well qualities can be predicted from per-90 stats.

In [ ]:
# For each quality, find its highest correlation with any per-90 metric
qual_vs_p90 = []

for qcol in qualities:
    qname = qcol.replace('from_', '')
    best_corr = 0
    best_p90 = ''
    for pcol in per90:
        mask = df[qcol].notna() & df[pcol].notna()
        if mask.sum() > 1000:
            r = df.loc[mask, qcol].corr(df.loc[mask, pcol])
            if abs(r) > abs(best_corr):
                best_corr = r
                best_p90 = pcol.replace('from_', '')
    qual_vs_p90.append({'quality': qname, 'best_per90': best_p90, 'corr': best_corr})

qvp = pd.DataFrame(qual_vs_p90).sort_values('corr', ascending=False, key=abs)
print('Each quality\'s strongest correlation with a per-90 metric:')
print('(High correlation = quality is largely captured by per-90 data)')
print('(Low correlation = quality adds unique information)')
print()
qvp.style.format({'corr': '{:+.3f}'}).background_gradient(cmap='RdYlGn', subset=['corr'], vmin=0, vmax=1)

## 8. Z-score vs Per-90: same metric, two representations

For the 67 metrics that exist in BOTH per-90 and z-score form: how correlated are they?  
If they're ~perfectly correlated, keeping both is redundant.

In [ ]:
# For each shared metric, correlate per-90 vs z-score
p90_zs_corr = []
for p90_col in per90:
    metric_name = p90_col.replace('from_', '')
    zs_col = f'from_z_score_{metric_name}'
    if zs_col in df.columns:
        mask = df[p90_col].notna() & df[zs_col].notna()
        if mask.sum() > 1000:
            r = df.loc[mask, p90_col].corr(df.loc[mask, zs_col])
            p90_zs_corr.append({'metric': metric_name, 'corr': r, 'n': mask.sum()})

pzc = pd.DataFrame(p90_zs_corr).sort_values('corr', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(pzc)), pzc['corr'], color='steelblue')
ax.set_yticks(range(len(pzc)))
ax.set_yticklabels(pzc['metric'], fontsize=6)
ax.axvline(0.9, color='red', ls='--', alpha=0.5, label='r=0.9')
ax.set_xlabel('Correlation (per-90 vs z-score)')
ax.set_title('Per-90 ↔ Z-score correlation for each metric\n(high = z-score is mostly a rescaled per-90)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Metrics with corr > 0.9: {(pzc["corr"] > 0.9).sum()} / {len(pzc)}')
print(f'Metrics with corr > 0.95: {(pzc["corr"] > 0.95).sum()} / {len(pzc)}')
print(f'\nLowest correlations (z-score adds most beyond per-90):')
pzc.head(10)

## 9. Structural columns quick check

In [ ]:
struct_cols = ['player_season_age', 'from_Minutes', 'to_Minutes',
               'from_position', 'to_position', 'from_season', 'to_season',
               'from_competition', 'to_competition',
               'from_matches_pct', 'to_matches_pct',
               'from_minutes_pct', 'to_minutes_pct']

for c in struct_cols:
    if c in df.columns:
        null = df[c].isnull().mean() * 100
        uniq = df[c].nunique()
        print(f'{c:<30} null={null:.1f}%  unique={uniq}')

## 10. Summary — decision framework

This is what the data tells us. **You decide.**

| Representation | Pros | Cons | Metrics |
|----------------|------|------|---------|
| **Qualities** (29) | Compact, capture multi-dimensional concepts | Black-box composites from Twelve, opaque formula | Finishing, Pressing, etc. |
| **Per-90** (67) | Concrete, interpretable, granular | Not position-normalized, scale varies wildly | Goals per 90, xG per 90, etc. |
| **Z-scores** (75) | Position-normalized, standardized scale | Cross-season baselines may shift (different reference populations) | Same as per-90 + a few extra |

### Overlap summary
- Z-scores cover all per-90 metrics **plus** some extras (8 quality-like metrics that got z-scored)
- Qualities are mostly predictable from per-90 data (check correlations above)
- Per-90 and z-scores for the same metric are heavily correlated (but z-scores adjust for position)

In [ ]:
# Final count summary
print('If you keep only Per-90:      67 from + 67 to = 134 player metric columns')
print('If you keep only Z-scores:    75 from + 75 to = 150 player metric columns')
print('If you keep only Qualities:   29 from + 29 to =  58 player metric columns')
print('If you keep Per-90 + Qual:    67+29 from + 67+29 to = 192 player metric columns')
print('If you keep all three:        171 from + 171 to = 342 player metric columns')
print()
print('Structural columns (always keep): ~16')
print('Team stats (separate notebook): 148')
print('External/orphan (separate notebook): 33')